In [1]:
import pandas as pd
import numpy as np
import joblib
import random
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

In [2]:
df = pd.read_csv("feature_enginn 1.0 dataset.csv")

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)
df.rename(
    columns={"Magnitue": "Magnitude"},
    inplace=True
)
df.fillna(0, inplace=True)
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (3380000, 47)


,flow_duration,Header_Length,Protocol_Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot_size,IAT,Number,Magnitude,Radius,Covariance,Variance,Weight,label
0,0.000000,180.18,16.84,64.0,15.758818,15.758818,0.0,0.0,0.0,0.0,...,0.563055,181.16,8.300745e+07,9.5,19.071659,0.802637,10.738677,0.03,141.55,21
1,0.000000,0.00,1.00,64.0,0.996082,0.996082,0.0,0.0,0.0,0.0,...,0.000000,42.00,8.314936e+07,9.5,9.165151,0.000000,0.000000,0.00,141.55,6
2,0.000000,54.00,6.00,64.0,0.718000,0.718000,0.0,0.0,1.0,0.0,...,0.000000,54.00,8.309409e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,10
3,0.000000,54.00,6.00,64.0,6.211557,6.211557,0.0,0.0,0.0,0.0,...,0.000000,54.00,8.303713e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,13
4,5.003695,114.98,6.11,64.0,0.418744,0.418744,0.0,0.0,0.0,0.0,...,0.026812,53.96,8.333211e+07,9.5,10.391685,0.038221,0.024351,0.03,141.55,8


In [3]:
from sklearn.preprocessing import LabelEncoder
min_samples = 300
counts = df['label'].value_counts()
valid_classes = counts[
    counts >= min_samples].index
df = df[
    df['label'].isin(valid_classes)
].reset_index(drop=True)
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
mapping_df = pd.DataFrame({
    "label": range(len(le.classes_)),
    "attack_name": le.classes_
})
mapping_df.to_csv("label_mapping.csv",index=False)
joblib.dump(le,"label_encoder.pkl")
print("Remaining Classes:",
      len(le.classes_))

Remaining Classes: 30


In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=SEED
)

train_df.reset_index(
    drop=True,
    inplace=True
)

test_df.reset_index(
    drop=True,
    inplace=True
)

y_train = train_df['label'].values
y_test = test_df['label'].values

print("Train Shape:",
      train_df.shape)

print("Test Shape:",
      test_df.shape)

Train Shape: (2703392, 47)
Test Shape: (675848, 47)


In [5]:
for df_ in [train_df, test_df]:
    df_['packet_rate'] = (df_['Tot_sum']/ (df_['Duration'] + 1))
    df_['byte_rate'] = (df_['Tot_size']/ (df_['Duration'] + 1))
    df_['packet_diff'] = (df_['Max']- df_['Min'])
    df_['packet_ratio'] = (df_['syn_count']/ (df_['ack_count'] + 1))
    df_['bytes_per_packet'] = (df_['Tot_size']/ (df_['Tot_sum'] + 1))
    df_['flow_intensity'] = (df_['Tot_size']/ (df_['Duration'] + 1))
    df_['stability'] = (df_['Std']/ (df_['AVG'] + 1))
    df_['pkt_irregularity'] = (df_['packet_diff']/ (df_['packet_rate'] + 1))
print("8 Behavioral Features Added")

8 Behavioral Features Added


In [6]:
min_samples = 200
counts = train_df['label'].value_counts()
valid_classes = counts[counts >= min_samples].index
train_df = train_df[train_df['label'].isin(valid_classes)].reset_index(drop=True)

test_df = test_df[test_df['label'].isin(valid_classes)].reset_index(drop=True)

y_train = train_df['label'].values
y_test = test_df['label'].values

print("Final Classes:",
      len(valid_classes))

Final Classes: 30


In [7]:
features = [
    'flow_duration',
    'Header_Length',
    'Protocol_Type',
    'Duration',
    'HTTP',
    'HTTPS',
    'DNS',
    'TCP',
    'UDP',
    'ICMP',
    'syn_flag_number',
    'ack_flag_number',
    'rst_flag_number',
    'ack_count',
    'syn_count',
    'rst_count',
    'Tot_sum',
    'Min',
    'Max',
    'AVG',
    'Std',
    'Tot_size',
    'Radius',
    'Covariance',
    'Variance',
    'Magnitude',
    'Weight',
    'packet_rate',
    'byte_rate',
    'packet_diff',
    'packet_ratio',
    'bytes_per_packet',
    'flow_intensity',
    'stability',
    'pkt_irregularity'
]

X_train = train_df[features]
X_test = test_df[features]

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

Training Features Shape: (2703392, 35)
Testing Features Shape: (675848, 35)


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

rf = RandomForestClassifier(
    n_estimators=120,
    max_depth=18,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_probs = rf.predict_proba(X_test)

rf_conf_max = rf_probs.max(axis=1)

print("RF Accuracy:",
      accuracy_score(
          y_test,
          rf_pred
      ))

print("RF Macro F1:",
      f1_score(
          y_test,
          rf_pred,
          average='macro'
      ))

RF Accuracy: 0.8785851256495544
RF Macro F1: 0.6952461680526283


In [9]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=250,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=1,
    reg_lambda=2,
    random_state=SEED,
    eval_metric='mlogloss',
    n_jobs=-1
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)

print("XGB Accuracy:",
      accuracy_score(
          y_test,
          xgb_pred
      ))

print("XGB Macro F1:",
      f1_score(
          y_test,
          xgb_pred,
          average='macro'
      ))

XGB Accuracy: 0.8884971176951031
XGB Macro F1: 0.7360541779755081


In [10]:
thresholds = np.arange(
    0.70,
    0.90,
    0.02
)

results = []

for t in thresholds:

    preds = [

        rf_pred[i]
        if rf_conf_max[i] >= t
        else xgb_pred[i]

        for i in range(len(X_test))
    ]

    acc = accuracy_score(
        y_test,
        preds
    )

    f1 = f1_score(
        y_test,
        preds,
        average='macro'
    )

    results.append(
        (t, acc, f1)
    )

results_df = pd.DataFrame(
    results,
    columns=[
        "Threshold",
        "Accuracy",
        "Macro_F1"
    ]
)

print(results_df)

    Threshold  Accuracy  Macro_F1
0        0.70  0.888207  0.736014
1        0.72  0.888309  0.736141
2        0.74  0.888370  0.736111
3        0.76  0.888425  0.736045
4        0.78  0.888441  0.736060
5        0.80  0.888459  0.736077
6        0.82  0.888468  0.736048
7        0.84  0.888490  0.736071
8        0.86  0.888497  0.736076
9        0.88  0.888497  0.736063
10       0.90  0.888497  0.736049


In [11]:
optimal_threshold = results_df.loc[
    results_df['Macro_F1'].idxmax(),
    'Threshold'
]

print("Optimal Threshold:",
      optimal_threshold)

Optimal Threshold: 0.72


In [12]:
final_pred = [

    rf_pred[i]
    if rf_conf_max[i] >= optimal_threshold
    else xgb_pred[i]

    for i in range(len(X_test))
]

print("Cascade Accuracy:",
      accuracy_score(
          y_test,
          final_pred
      ))

print("Cascade Macro F1:",
      f1_score(
          y_test,
          final_pred,
          average='macro'
      ))

Cascade Accuracy: 0.8883092056201987
Cascade Macro F1: 0.7361411033476354


Stage 2: XGBoost (Full Data Training)

Cascade Model (RF + XGBoost)

In [13]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        final_pred
    )
)

              precision    recall  f1-score   support

           0       0.81      0.96      0.88     15976
           1       0.94      0.18      0.30        84
           2       0.75      0.20      0.32        75
           3       0.98      0.99      0.99      4118
           4       0.87      0.82      0.85       421
           5       1.00      1.00      1.00    104496
           6       0.98      0.99      0.98      6526
           7       1.00      1.00      1.00     59193
           8       1.00      1.00      1.00     58498
           9       0.69      0.97      0.80     58829
          10       0.64      0.86      0.74       333
          11       0.99      0.72      0.83     51959
          12       0.79      0.98      0.87     65165
          13       0.86      0.90      0.88     78427
          14       0.99      0.99      0.99      4142
          15       0.57      0.46      0.51      2592
          16       0.60      0.13      0.21       193
          17       0.91    

In [14]:
for df_ in [train_df, test_df]:

    df_['rate_change'] = (
        df_['Rate']
        .diff()
        .fillna(0)
    )

    df_['iat_change'] = (
        df_['IAT']
        .diff()
        .fillna(0)
    )

    df_['size_change'] = (
        df_['Tot_size']
        .diff()
        .fillna(0)
    )

    df_['rolling_rate'] = (
        df_['Rate']
        .rolling(5)
        .mean()
        .fillna(0)
    )

print("GRU Features Created")

GRU Features Created


In [15]:
from sklearn.preprocessing import StandardScaler

gru_features = [

    'IAT',
    'Rate',
    'Srate',
    'Drate',

    'flow_duration',
    'Tot_size',
    'AVG',
    'Std',
    'Header_Length',

    'rate_change',
    'iat_change',
    'size_change',
    'rolling_rate'
]

scaler = StandardScaler()

train_df[gru_features] = (
    scaler.fit_transform(
        train_df[gru_features]
    )
)

test_df[gru_features] = (
    scaler.transform(
        test_df[gru_features]
    )
)

joblib.dump(
    scaler,
    "gru_scaler.pkl"
)

print("Scaling Complete")

Scaling Complete


In [16]:
train_df['cum_time'] = (
    train_df['IAT'].cumsum()
)

test_df['cum_time'] = (
    test_df['IAT'].cumsum()
)

bucket_size = 1000

train_df['time_bucket'] = (
    train_df['cum_time']
    // bucket_size
).astype(int)

test_df['time_bucket'] = (
    test_df['cum_time']
    // bucket_size
).astype(int)

train_df['session'] = (

    train_df['Protocol_Type']
    .astype(str)

    + "_"

    + train_df['time_bucket']
    .astype(str)
)

test_df['session'] = (

    test_df['Protocol_Type']
    .astype(str)

    + "_"

    + test_df['time_bucket']
    .astype(str)
)

print("Sessions Created")

Sessions Created


In [17]:
from collections import Counter

def create_sequences(
    df,
    features,
    seq_len=10
):

    X = []
    y = []

    for _, group in df.groupby('session'):

        group = group.sort_values(
            by='IAT'
        )

        data = group[
            features
        ].values

        labels = group[
            'label'
        ].values

        for i in range(len(data)):

            start = max(
                0,
                i - seq_len + 1
            )

            seq = data[start:i+1]

            if len(seq) < seq_len:

                pad = np.zeros(
                    (
                        seq_len - len(seq),
                        data.shape[1]
                    )
                )

                seq = np.vstack(
                    (pad, seq)
                )

            else:

                seq = seq[-seq_len:]

            label = Counter(
                labels[start:i+1]
            ).most_common(1)[0][0]

            X.append(seq)
            y.append(label)

    return (
        np.array(X, dtype=np.float32),
        np.array(y)
    )

In [ ]:
X_gru_seq_train, y_gru_seq_train = (
    create_sequences(
        train_df,
        gru_features
    )
)

X_gru_seq_test, y_gru_seq_test = (
    create_sequences(
        test_df,
        gru_features
    )
)

print(
    "Train Shape:",
    X_gru_seq_train.shape
)

print(
    "Test Shape:",
    X_gru_seq_test.shape
)



Train Shape: (2703392, 10, 13)
Test Shape: (675848, 10, 13)


In [19]:
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    GRU,
    Dense,
    Dropout,
    Bidirectional
)

from sklearn.utils.class_weight import (
    compute_class_weight
)

classes = np.unique(
    y_gru_seq_train
)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_gru_seq_train
)

class_weights = dict(
    zip(classes, weights)
)

gru_model = Sequential([

    Bidirectional(

        GRU(
            128,
            return_sequences=True
        ),

        input_shape=(
            X_gru_seq_train.shape[1],
            X_gru_seq_train.shape[2]
        )
    ),

    Dropout(0.3),

    Bidirectional(
        GRU(64)
    ),

    Dropout(0.3),

    Dense(
        64,
        activation='relu'
    ),

    Dense(
        len(classes),
        activation='softmax'
    )
])

gru_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

gru_model.summary()

C:\Users\avihs\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 10, 256)        │       109,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       123,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 30)             │         1,950 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243,678 (951.87 KB)

 Trainable params: 243,678 (951.87 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
history = gru_model.fit(

    X_gru_seq_train,
    y_gru_seq_train,

    validation_split=0.2,

    epochs=20,

    batch_size=64,

    class_weight=class_weights,

    verbose=1
)

Epoch 1/20
33793/33793 ━━━━━━━━━━━━━━━━━━━━ 916s 27ms/step - accuracy: 0.7310 - loss: 1.0585 - val_accuracy: 0.7322 - val_loss: 0.6441
Epoch 2/20
33793/33793 ━━━━━━━━━━━━━━━━━━━━ 1036s 31ms/step - accuracy: 0.9042 - loss: 0.8851 - val_accuracy: 0.8350 - val_loss: 0.4414
Epoch 3/20
33793/33793 ━━━━━━━━━━━━━━━━━━━━ 1027s 30ms/step - accuracy: 0.9250 - loss: 0.8920 - val_accuracy: 0.8688 - val_loss: 0.3644
Epoch 4/20
33793/33793 ━━━━━━━━━━━━━━━━━━━━ 988s 29ms/step - accuracy: 0.9393 - loss: 0.8426 - val_accuracy: 0.8842 - val_loss: 0.3307
Epoch 5/20
33793/33793 ━━━━━━━━━━━━━━━━━━━━ 1000s 30ms/step - accuracy: 0.9426 - loss: 0.8793 - val_accuracy: 0.8916 - val_loss: 0.3878
Epoch 6/20
33793/33793 ━━━━━━━━━━━━━━━━━━━━ 947s 28ms/step - accuracy: 0.9424 - loss: 1.0952 - val_accuracy: 0.8923 - val_loss: 0.3500
Epoch 7/20
33793/33793 ━━━━━━━━━━━━━━━━━━━━ 699s 21ms/step - accuracy: 0.9418 - loss: 1.1397 - val_accuracy: 0.7097 - val_loss: 0.7370
Epoch 8/20
33793/33793 ━━━━━━━━━━━━━━━━━━━━ 478s 14m

In [21]:
gru_pred = np.argmax(

    gru_model.predict(
        X_gru_seq_test
    ),

    axis=1
)

print(
    "GRU Accuracy:",
    accuracy_score(
        y_gru_seq_test,
        gru_pred
    )
)

print(
    "GRU Macro F1:",
    f1_score(
        y_gru_seq_test,
        gru_pred,
        average='macro'
    )
)

21121/21121 ━━━━━━━━━━━━━━━━━━━━ 85s 4ms/step
GRU Accuracy: 0.9319343994507641
GRU Macro F1: 0.5526407302070561


In [22]:
joblib.dump(
    rf,
    "rf_model.pkl"
)

joblib.dump(
    xgb,
    "xgb_model.pkl"
)

gru_model.save(
    "gru_model.keras"
)

print("All Models Saved")

All Models Saved


In [23]:
from sklearn.metrics import (
    precision_score,
    recall_score
)

comparison = pd.DataFrame({

    "Model": [
        "RF",
        "XGB",
        "Cascade",
        "GRU"
    ],

    "Accuracy": [

        accuracy_score(
            y_test,
            rf_pred
        ),

        accuracy_score(
            y_test,
            xgb_pred
        ),

        accuracy_score(
            y_test,
            final_pred
        ),

        accuracy_score(
            y_gru_seq_test,
            gru_pred
        )
    ],

    "Precision": [

        precision_score(
            y_test,
            rf_pred,
            average='macro'
        ),

        precision_score(
            y_test,
            xgb_pred,
            average='macro'
        ),

        precision_score(
            y_test,
            final_pred,
            average='macro'
        ),

        precision_score(
            y_gru_seq_test,
            gru_pred,
            average='macro'
        )
    ],

    "Recall": [

        recall_score(
            y_test,
            rf_pred,
            average='macro'
        ),

        recall_score(
            y_test,
            xgb_pred,
            average='macro'
        ),

        recall_score(
            y_test,
            final_pred,
            average='macro'
        ),

        recall_score(
            y_gru_seq_test,
            gru_pred,
            average='macro'
        )
    ],

    "F1": [

        f1_score(
            y_test,
            rf_pred,
            average='macro'
        ),

        f1_score(
            y_test,
            xgb_pred,
            average='macro'
        ),

        f1_score(
            y_test,
            final_pred,
            average='macro'
        ),

        f1_score(
            y_gru_seq_test,
            gru_pred,
            average='macro'
        )
    ]
})

print(comparison)

comparison.to_csv(
    "model_comparison.csv",
    index=False
)

     Model  Accuracy  Precision    Recall        F1
0       RF  0.878585   0.686613  0.735862  0.695246
1      XGB  0.888497   0.808214  0.720831  0.736054
2  Cascade  0.888309   0.806760  0.721868  0.736141
3      GRU  0.931934   0.561040  0.652820  0.552641
